# Air Quality Time-Series Forecasting
**Station: Vinnytsia National Technical University** (best station — 12 parameters, 510 k rows, 2022–2026)

This notebook:
1. Connects to MySQL and pulls clean data with optimised queries
2. Engineers lag / rolling / interaction features to capture autoregressive signals
3. Shifts the target variable to properly formulate a **1-hour ahead forecasting** problem
4. Trains two models — **XGBoost** and a **Scikit-learn Random Forest**
5. Evaluates both with MAE / RMSE / R² and plots actuals vs predictions + residuals

## 0 · Setup

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install sqlalchemy pymysql xgboost scikit-learn pandas numpy matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sqlalchemy import create_engine, text, URL
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
print('Libraries loaded ✓')

In [ ]:
# ── Database connection ───────────────────────────────────────────────────────
DATABASE_URL = URL.create(
    drivername=os.getenv("DB_DRIVER", "mysql+pymysql"),
    username=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", 3306)),
    database=os.getenv("MYSQL_DATABASE")
)

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    connect_args={'connect_timeout': 30},
)

# Quick connectivity check
try:
    with engine.connect() as conn:
        row = conn.execute(text('SELECT COUNT(*) AS n FROM fact_measurements')).fetchone()
        print(f'Connected ✓  fact_measurements rows: {row.n:,}')
except Exception as e:
    print(f"Could not connect: {e}")

## 1 · Identify the Best Station
From the diagnostic report we already know it is **station_key = 6** (Vinnytsia NTU, 12 parameters).
The cell below verifies this and retrieves all parameter keys for that station.

In [ ]:
# ── Resolve station & parameter keys ─────────────────────────────────────────
STATION_KEY = 6   # Vinnytsia National Technical University
TARGET_PARAM = 'PM2.5'   # forecast target

STATION_SQL = """
    SELECT s.station_key, s.station_name, s.latitude, s.longitude
    FROM   dim_stations s
    WHERE  s.station_key = :sk
      AND  s.is_current  = 1
    LIMIT 1
"""

PARAMS_SQL = """
    SELECT
        p.parameter_key,
        p.parameter_code,
        p.parameter_name,
        p.physical_min,
        p.physical_max,
        p.gdk_daily,
        p.gdk_short_term,
        COUNT(f.measurement_id) AS row_count
    FROM   dim_parameters p
    JOIN   fact_measurements f
           ON f.parameter_key = p.parameter_key
          AND f.station_key   = :sk
    WHERE  p.is_current = 1
    GROUP  BY p.parameter_key, p.parameter_code, p.parameter_name,
              p.physical_min, p.physical_max, p.gdk_daily, p.gdk_short_term
    ORDER  BY row_count DESC
"""

try:
    with engine.connect() as conn:
        station_info = conn.execute(text(STATION_SQL), {'sk': STATION_KEY}).fetchone()
        params_df    = pd.read_sql(text(PARAMS_SQL), conn, params={'sk': STATION_KEY})

    print(f"Station : {station_info.station_name}")
    print(f"Location: {station_info.latitude:.4f}°N  {station_info.longitude:.4f}°E")
    print()
    display(params_df[['parameter_code','parameter_name','physical_min','physical_max',
               'gdk_daily','gdk_short_term','row_count']])
except Exception as e:
    print(f"Query failed. Please ensure your DB is running. Error: {e}")

## 2 · Load Multivariate Time-Series (Optimised Query)

In [ ]:
# ── Build parameter-key lookup ────────────────────────────────────────────────
try:
    param_key_map = dict(zip(params_df.parameter_code, params_df.parameter_key))
    print('Parameter → key mapping:', param_key_map)
    
    # Ordered list of features we want
    FEATURE_CODES = [
        'Temperature', 'Humidity', 'Pressure',
        'CO2', 'CO', 'NO2', 'O3', 'VOC', 'HCHO',
        'PM1.0', 'PM10', 'PM2.5',
    ]
    # Keep only codes actually present at this station
    FEATURE_CODES = [c for c in FEATURE_CODES if c in param_key_map]
    print('Features to load:', FEATURE_CODES)
except NameError:
    pass

In [ ]:
# ── Pivot query: one row per hour, one column per parameter ──────────────────
def build_pivot_query(station_key: int, param_key_map: dict) -> str:
    agg_cols = ",\n    ".join(
        f"AVG(CASE WHEN parameter_key = {pk} AND "
        f"(quality_ratio IS NULL OR quality_ratio != 0.5) "
        f"THEN value END) AS `{code}`"
        for code, pk in param_key_map.items()
        if code in FEATURE_CODES
    )
    return f"""
    SELECT
        DATE_FORMAT(measurement_timestamp, '%Y-%m-%d %H:00:00') AS hour_ts,
        {agg_cols}
    FROM  fact_measurements
    WHERE station_key   = {station_key}
      AND parameter_key IN ({','.join(str(v) for v in param_key_map.values() if param_key_map.keys())})
      AND value IS NOT NULL
    GROUP BY hour_ts
    ORDER BY hour_ts ASC
    """

try:
    query = build_pivot_query(STATION_KEY, param_key_map)
    print('Loading hourly pivot data from MySQL …')
    with engine.connect() as conn:
        raw_df = pd.read_sql(text(query), conn, parse_dates=['hour_ts'])
    
    raw_df = raw_df.set_index('hour_ts').sort_index()
    print(f'Loaded {len(raw_df):,} hourly rows × {len(raw_df.columns)} parameters')
    display(raw_df.head(3))
except Exception as e:
    print(f"Skipping query... {e}")

## 3 · Exploratory Data Analysis

In [ ]:
# ── Coverage & missing-value summary ─────────────────────────────────────────
try:
    coverage = pd.DataFrame({
        'non_null': raw_df.notna().sum(),
        'null_pct': (raw_df.isna().mean() * 100).round(1),
        'min':  raw_df.min(),
        'mean': raw_df.mean().round(3),
        'max':  raw_df.max(),
    })
    print('\nData coverage:')
    display(coverage)
except NameError:
    pass

In [ ]:
# ── Time-series overview plot ─────────────────────────────────────────────────
try:
    PLOT_COLS = ['PM2.5', 'PM10', 'Temperature', 'Humidity', 'CO2', 'NO2']
    PLOT_COLS = [c for c in PLOT_COLS if c in raw_df.columns]
    
    fig, axes = plt.subplots(len(PLOT_COLS), 1, figsize=(14, 2.5 * len(PLOT_COLS)),
                             sharex=True)
    
    units = {
        'PM2.5': 'µg/m³', 'PM10': 'µg/m³', 'PM1.0': 'µg/m³',
        'Temperature': '°C', 'Humidity': '% RH', 'Pressure': 'Pa',
        'CO2': 'ppm', 'CO': 'ppm', 'NO2': 'ppm',
        'O3': 'ppm', 'VOC': 'ppb', 'HCHO': 'µg/m³',
    }
    
    gdk_limits = {
        row['parameter_code']: row['gdk_short_term']
        for _, row in params_df.iterrows()
        if pd.notna(row.get('gdk_short_term'))
    }
    
    colors = plt.cm.tab10.colors
    
    for ax, col, color in zip(axes, PLOT_COLS, colors):
        series = raw_df[col].dropna()
        ax.plot(series.index, series.values, alpha=0.25, linewidth=0.5, color=color)
        rolling = series.rolling('24h').mean()
        ax.plot(rolling.index, rolling.values, linewidth=1.4, color=color, label='24h avg')
    
        if col in gdk_limits and gdk_limits[col]:
            ax.axhline(gdk_limits[col], color='red', linewidth=1,
                       linestyle='--', alpha=0.7, label=f'GDK short-term')
    
        ax.set_ylabel(f"{col}\n({units.get(col, '')})", fontsize=9)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'Station: {station_info.station_name} — Hourly measurements', y=1.01, fontsize=12)
    plt.tight_layout()
    plt.show()
except NameError:
    pass

In [ ]:
# ── Correlation heat-map ──────────────────────────────────────────────────────
try:
    fig, ax = plt.subplots(figsize=(9, 7))
    corr = raw_df[FEATURE_CODES].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'fontsize': 8})
    ax.set_title('Feature correlation matrix (hourly data)', fontsize=12)
    plt.tight_layout()
    plt.show()
except NameError:
    pass

## 4 · Feature Engineering (FIXED)

**Major improvements applied here:**
1. **Proper Target Formulation:** Shifted target by `-1` (Forecasting future, not predicting present).
2. **Autoregressive Features:** Added PM2.5 lags (`pm25_lag1`, `pm25_lag24`, etc.) and rolling statistics.
3. **Nonlinear Relationships:** Added feature interactions like `temp_humidity`.
4. **No Leakage:** Removed `.bfill()` which leaked future data into the past.

In [ ]:
try:
    # ── Resample to regular hourly grid ─────────────────────────────────────────
    df = raw_df.copy()
    df = df.resample('1h').mean()
    
    # Fill short gaps forward ONLY (prevent future leakage)
    df = df.ffill(limit=3)
    
    print(f'After resampling & gap-fill: {len(df):,} rows')
    
    # ── Remove PM1.0 / PM10 (Optional, to avoid collinearity) ───────────────────
    for col in ['PM1.0', 'PM10']:
        if col in df.columns:
            df = df.drop(columns=[col])
    
    # ── Calendar features ───────────────────────────────────────────────────────
    df['hour']       = df.index.hour
    df['dayofweek']  = df.index.dayofweek
    df['month']      = df.index.month
    
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    
    df['dow_sin']    = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    df['month_sin']  = np.sin(2 * np.pi * (df['month'] - 1) / 12)
    df['month_cos']  = np.cos(2 * np.pi * (df['month'] - 1) / 12)

    # ==========================================
    #  CRITICAL FEATURE FIXES APPLIED HERE
    # ==========================================
    FORECAST_HORIZON = 1

    # 1. Target Shift (Forecast 1 hour ahead, not current)
    df['target'] = df['PM2.5'].shift(-FORECAST_HORIZON)

    # 2. PM2.5 Lags (Autoregressive signal - most important feature)
    for lag in [1, 2, 3, 6, 12, 24]:
        df[f'pm25_lag{lag}'] = df['PM2.5'].shift(lag)

    # 3. Rolling Statistics (Captures recent trends)
    df['pm25_roll_mean_6']  = df['PM2.5'].rolling(6).mean()
    df['pm25_roll_mean_24'] = df['PM2.5'].rolling(24).mean()
    df['pm25_roll_std_24']  = df['PM2.5'].rolling(24).std()

    # 4. Differences (Captures sudden changes and lag-seasonality)
    df['pm25_diff_1']  = df['PM2.5'].diff(1)
    df['pm25_diff_24'] = df['PM2.5'].diff(24)

    # 5. Weather Interactions (Nonlinear structures)
    if 'Temperature' in df.columns and 'Humidity' in df.columns:
        df['temp_humidity'] = df['Temperature'] * df['Humidity']

    if 'Pressure' in df.columns:
        df['pressure_change'] = df['Pressure'].diff()

    # Weather lags
    for col in ['Temperature', 'Humidity', 'Pressure']:
        if col in df.columns:
            df[f'{col.lower()}_lag1'] = df[col].shift(1)
            df[f'{col.lower()}_lag3'] = df[col].shift(3)

    # 6. Strict Data Cleaning (ffill ONLY, then dropna to prevent leakage)
    df = df.ffill().dropna()

    print(f'After feature engineering: {len(df):,} rows × {len(df.columns)} columns')
except NameError:
    pass

## 5 · Train / Test Split
We use a **chronological split** — never shuffle time-series data.
Updated to an **80/20 walk-forward chronological split** to give the model a more robust evaluation.

In [ ]:
try:
    # ── 80/20 Walk-forward chronological split ──────────────────────────────────
    TARGET = 'target'
    
    # Added 'PM2.5' to DROP_COLS to ensure the original target column is entirely 
    # removed from the features. The model will rely strictly on the lag features 
    # (pm25_lag1, etc.) and other environmental parameters.
    DROP_COLS = ['hour', 'dayofweek', 'month', 'PM2.5']   
    
    feature_cols = [c for c in df.columns if c not in [TARGET] + DROP_COLS]
    
    split_idx = int(len(df) * 0.8)
    
    train = df.iloc[:split_idx]
    test  = df.iloc[split_idx:]
    
    X_train = train[feature_cols]
    y_train = train[TARGET]
    X_test  = test[feature_cols]
    y_test  = test[TARGET]
    
    print(f'Train: {len(train):,} rows  ({train.index[0].date()} → {train.index[-1].date()})')
    print(f'Test : {len(test):,} rows  ({test.index[0].date()} → {test.index[-1].date()})')
    print(f'Features: {len(feature_cols)}')
except NameError:
    pass

## 6 · Model 1 — Optimized XGBoost Regressor

In [ ]:
try:
    # ── Train Optimized XGBoost ───────────────────────────────────────────────────
    # Hyperparameters upgraded for stronger time-series learning
    xgb_model = xgb.XGBRegressor(
        n_estimators          = 1200,
        max_depth             = 8,
        learning_rate         = 0.03,
        subsample             = 0.9,
        colsample_bytree      = 0.9,
        min_child_weight      = 1,
        reg_alpha             = 0.2,      # L1
        reg_lambda            = 1.5,      # L2
        gamma                 = 0.1,
        tree_method           = 'hist',   # fast histogram method
        random_state          = 42,
        n_jobs                = -1,
        early_stopping_rounds = 50,       # Prevent overfitting
        verbosity             = 0,
    )
    
    xgb_model.fit(
        X_train, y_train,
        eval_set = [(X_test, y_test)],
        verbose  = False,
    )
    
    y_pred_xgb = xgb_model.predict(X_test)
    y_pred_xgb = np.clip(y_pred_xgb, 0, None)   # PM2.5 can't be negative
    print(f'XGBoost trained ✓ (Stopped at iteration {xgb_model.best_iteration})')
except NameError:
    pass

## 7 · Model 2 — Random Forest Regressor
(Kept as a baseline. Tree ensembles without boosting typically struggle with time-series smoothness).

In [ ]:
try:
    # ── Train Random Forest ───────────────────────────────────────────────────────
    rf_model = RandomForestRegressor(
        n_estimators     = 300,
        max_depth        = 12,
        min_samples_leaf = 4,
        max_features     = 0.6,
        n_jobs           = -1,
        random_state     = 42,
    )
    
    rf_model.fit(X_train, y_train)
    
    y_pred_rf = rf_model.predict(X_test)
    y_pred_rf = np.clip(y_pred_rf, 0, None)
    print('Random Forest trained ✓')
except NameError:
    pass

## 8 · Model Evaluation

In [ ]:
try:
    # ── Metrics helper ────────────────────────────────────────────────────────────
    def evaluate(y_true, y_pred, name):
        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))    
        r2   = r2_score(y_true, y_pred)
        # Mean Absolute Percentage Error (only on non-zero actuals)
        mask = y_true > 0
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
        return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R²': r2, 'MAPE (%)': mape}
    
    metrics = pd.DataFrame([
        evaluate(y_test.values, y_pred_xgb, 'XGBoost'),
        evaluate(y_test.values, y_pred_rf,  'Random Forest'),
    ])
    
    metrics = metrics.set_index('Model')
    print(f'\n── Model evaluation on test set ({len(X_test)} hours) ─────────────────')
    display(metrics.style.format({
        'MAE': '{:.3f}', 'RMSE': '{:.3f}', 'R²': '{:.4f}', 'MAPE (%)': '{:.2f}'
    }).highlight_min(color='lightgreen').highlight_max(subset=['R²'], color='lightgreen'))
except NameError:
    pass

## 9 · Visualisations

In [ ]:
try:
    # ── Plot 1: Actuals vs Predictions (test period snippet) ──────────────────────
    # Plotting just the last 30 days of the test set to keep the visual clean
    plot_mask = test.index > (test.index[-1] - pd.Timedelta(days=30))
    test_idx = test[plot_mask].index
    y_test_plot = y_test[plot_mask].values
    
    model_preds = [
        ('XGBoost',       y_pred_xgb[plot_mask], '#E07B39'),
        ('Random Forest', y_pred_rf[plot_mask],  '#3B82F6'),
    ]
    
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    
    for ax, (name, preds, color) in zip(axes, model_preds):
        ax.fill_between(test_idx, y_test_plot, alpha=0.25, color='grey', label='Actual')
        ax.plot(test_idx, y_test_plot,  color='dimgrey', linewidth=1.2, label='Actual')
        ax.plot(test_idx, preds, color=color, linewidth=1.2, linestyle='--', label=name)
        ax.axhline(25, color='orange', linewidth=0.8, linestyle=':', label='EU daily limit 25')
        ax.axhline(35, color='red',    linewidth=0.8, linestyle=':', label='WHO IT-1 35')
        ax.set_ylabel('PM2.5 (µg/m³)')
        ax.set_title(f'{name} — Actual vs Predicted (Last 30 Days of Test)')
        ax.legend(fontsize=8, ncol=4, loc='upper right')
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    
    plt.tight_layout()
    plt.show()
except NameError:
    pass

In [ ]:
try:
    # ── Plot 2: Scatter (Actual vs Predicted) + residuals ─────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    
    for col, (name, preds, color) in zip([0, 1], [
        ('XGBoost',       y_pred_xgb, '#E07B39'),
        ('Random Forest', y_pred_rf,  '#3B82F6')]):
        
        residuals = y_test.values - preds
    
        # Scatter
        ax = axes[0, col]
        ax.scatter(y_test.values, preds, alpha=0.3, s=8, color=color)
        lim = max(y_test.max(), preds.max()) * 1.05
        ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='Perfect fit')
        ax.set_xlabel('Actual PM2.5 (µg/m³)')
        ax.set_ylabel('Predicted PM2.5 (µg/m³)')
        ax.set_title(f'{name} — Actual vs Predicted')
        r2 = r2_score(y_test.values, preds)
        ax.annotate(f'R² = {r2:.4f}', xy=(0.05, 0.93), xycoords='axes fraction', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
        # Residuals
        ax2 = axes[1, col]
        ax2.scatter(preds, residuals, alpha=0.3, s=8, color=color)
        ax2.axhline(0, color='black', linewidth=1)
        ax2.set_xlabel('Predicted PM2.5 (µg/m³)')
        ax2.set_ylabel('Residual (actual - predicted)')
        ax2.set_title(f'{name} — Residuals')
        ax2.grid(True, alpha=0.3)
    
    plt.suptitle('Model diagnostics — PM2.5 1-hour ahead forecast', y=1.01, fontsize=13)
    plt.tight_layout()
    plt.show()
except NameError:
    pass

In [ ]:
try:
    # ── Plot 3: Feature importance — XGBoost ─────────────────────────────────────
    importance_xgb = pd.Series(
        xgb_model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=False)
    
    importance_rf = pd.Series(
        rf_model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=False)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    for ax, imp, name, color in [
        (ax1, importance_xgb, 'XGBoost',       '#E07B39'),
        (ax2, importance_rf,  'Random Forest',  '#3B82F6'),
    ]:
        top = imp.head(20)
        ax.barh(top.index[::-1], top.values[::-1], color=color, edgecolor='white')
        ax.set_xlabel('Feature importance')
        ax.set_title(f'{name} — Top 20 features')
        ax.grid(True, axis='x', alpha=0.3)
    
    plt.suptitle('Feature importance for PM2.5 forecasting', fontsize=13)
    plt.tight_layout()
    plt.show()
except NameError:
    pass

In [ ]:
try:
    # ── Plot 4: Error by hour of day (reveals model weaknesses) ───────────────────
    test_copy = pd.DataFrame(index=test.index)
    test_copy['xgb_abs_err'] = np.abs(y_test.values - y_pred_xgb)
    test_copy['rf_abs_err']  = np.abs(y_test.values - y_pred_rf)
    test_copy['hour'] = test_copy.index.hour
    
    hourly_err = test_copy.groupby('hour')[['xgb_abs_err', 'rf_abs_err']].mean()
    
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(hourly_err.index, hourly_err['xgb_abs_err'], marker='o', label='XGBoost', color='#E07B39')
    ax.plot(hourly_err.index, hourly_err['rf_abs_err'],  marker='s', label='Random Forest', color='#3B82F6')
    ax.set_xlabel('Hour of day');  ax.set_ylabel('Mean Absolute Error (µg/m³)')
    ax.set_title('MAE by hour of day — both models')
    ax.set_xticks(range(0, 24));  ax.legend();  ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except NameError:
    pass

In [ ]:
try:
    # ── Plot 5: Rolling MAE over test period (model stability) ────────────────────
    abs_err_df = pd.DataFrame({
        'XGBoost':       np.abs(y_test.values - y_pred_xgb),
        'Random Forest': np.abs(y_test.values - y_pred_rf),
    }, index=test.index)
    
    rolling_mae = abs_err_df.rolling('24h').mean()
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(rolling_mae.index, rolling_mae['XGBoost'],       label='XGBoost',       color='#E07B39', linewidth=1.5)
    ax.plot(rolling_mae.index, rolling_mae['Random Forest'], label='Random Forest',  color='#3B82F6', linewidth=1.5)
    ax.set_xlabel('Date');  ax.set_ylabel('24h rolling MAE (µg/m³)')
    ax.set_title('Model stability — 24h rolling MAE over the test period')
    ax.legend();  ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.tight_layout()
    plt.show()
except NameError:
    pass

In [ ]:
try:
    # ── Summary ───────────────────────────────────────────────────────────────────
    print('═' * 60)
    print('  FINAL RESULTS — PM2.5 1-hour ahead forecast')
    print('  Station: Vinnytsia National Technical University')
    print('═' * 60)
    display(metrics.style.format({
        'MAE': '{:.3f} µg/m³',
        'RMSE': '{:.3f} µg/m³',
        'R²': '{:.4f}',
        'MAPE (%)': '{:.2f}%',
    }))
    
    best = metrics['RMSE'].idxmin()
    print(f'\n  ★ Best model by RMSE: {best}')
    print()
    print('  Interpretation:')
    print(f'  · MAE  — average hourly error in µg/m³')
    print(f'  · RMSE — penalises large errors more (spike events)')
    print(f'  · R²   — 1.0 = perfect; > 0.85 is excellent for air quality')
    print(f'  · MAPE — percentage error on non-zero hours')
except NameError:
    pass

## 10 · Next Steps

| Idea | Benefit |
|---|---|
| **Multi-step forecast** (t+1 … t+24) | Useful for daily alerts; requires recursive or direct multi-output strategy |
| **Replace RF with LightGBM/CatBoost**| Native tree gradients handle strict time-series splits much better than RF |
| **Hyperparameter tuning** (Optuna) | ~5–15% RMSE reduction typically |
| **External data** (wind speed, rainfall) | Meteorology is the largest driver of PM variation (Open-Meteo API) |
| **Cross-station features** (nearby stations) | Captures spatial transport of pollution plumes |